# Pipeline completo de src (6 etapas)

Este notebook ejecuta el pipeline oficial del proyecto usando exclusivamente funciones del codigo fuente en src, sin reescribir la logica matematica.

## Objetivo metodologico

1. Mostrar como se transforma una ecuacion parametrica F(x; theta)=0 en expresiones analiticas de sus ramas de raices.
2. Documentar cada etapa con trazabilidad: que entra, que funcion se llama, que sale y como se usa en la etapa siguiente.
3. Generar evidencia reproducible para entrega de tesis, con visualizacion de datos intermedios.

## Flujo completo

1. Etapa 1: parsear la ecuacion simbolica con SymPy.
2. Etapa 2: construir el grid cartesiano de parametros.
3. Etapa 3: resolver numericamente la ecuacion para cada tupla de parametros.
4. Etapa 4: agrupar raices por rama (orden estadistico/ordinal).
5. Etapa 5: ajustar expresiones por regresion simbolica en cada rama.
6. Etapa 6: consolidar expresiones finales (funcion unica o pieza por regiones).

## Criterios de calidad de este notebook

- No se inventan funciones nuevas: se importa desde src.
- La configuracion vigente se toma desde config.py.
- En cada etapa se imprime informacion suficiente para auditoria tecnica.
- La salida debe permitir justificar resultados en el informe de tesis.

## Setup: imports, paths y contrato de ejecucion

### Que hace este bloque

Este bloque prepara el entorno para que el notebook use el codigo real del proyecto.

1. Detecta automaticamente la raiz del proyecto buscando src/config.py y src/main.py.
2. Inserta src y sus submodulos en sys.path para habilitar imports directos.
3. Importa config y las funciones principales de las 6 etapas.
4. Imprime la configuracion activa para verificar trazabilidad experimental.

### Por que es importante

- Garantiza portabilidad: no depende de rutas absolutas de una maquina especifica.
- Evita duplicacion de logica: el notebook se comporta como un cliente del pipeline oficial.
- Facilita reproducibilidad: deja visible la ecuacion, variables, parametros y settings de iteracion.

### Salida esperada

Al ejecutar la celda deben verse: Project root, equation string, variables, parametros y parametros clave de iteracion (NITERATIONS y MAX_ITERATIONS). Si alguno no aparece, el entorno no quedo correctamente configurado.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import sympy as sp

# Mostrar tablas/arreglos completos en todo el notebook.
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
np.set_printoptions(threshold=np.inf, linewidth=200)


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'src' / 'config.py').exists() and (candidate / 'src' / 'main.py').exists():
            return candidate
    raise FileNotFoundError('No se encontro la raiz del proyecto con src/config.py y src/main.py')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for subdir in [
    '1_equation_definition',
    '2_parameter_grid',
    '3_zero_finding',
    '4_data_preparation',
    '5_symbolic_regression',
    '6_expression_builder',
]:
    p = SRC_DIR / subdir
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import config as cfg
from equation_parser import parse_equation
from grid_generator import generate_grid
from solver import solve_for_all_parameter_tuples
from root_grouping import group_by_root_branch
from regression_adapter import run_for_all_branches
from piecewise_builder import build_piecewise_expression, find_region_boundaries, simplify_to_single_expression

print('Project root:', PROJECT_ROOT)
print('Equation string:', cfg.EQUATION_STRING)
print('Variables:', cfg.VARIABLES)
print('Parametros:', cfg.PARAMETERS)
print('NITERATIONS:', cfg.NITERATIONS)
print('MAX_ITERATIONS (wrapper):', cfg.MAX_ITERATIONS)

Project root: /home/noel/Disco D/4to_Anno/tesis/Tesis
Equation string: x + a - 2
Variables: ['x']
Parametros: ['a']
NITERATIONS: 500
MAX_ITERATIONS (wrapper): None


## Etapa 1: Definicion simbolica de la ecuacion

### Objetivo

Convertir la ecuacion escrita como texto en una representacion simbolica formal que pueda manipularse algebraicamente.

### Entrada

- cfg.EQUATION_STRING: expresion F(x; theta).
- cfg.VARIABLES: variables de estado o desconocidas (ej. x).
- cfg.PARAMETERS: parametros del problema (ej. a, b, c).

### Proceso

La funcion parse_equation crea simbolos de SymPy para variables y parametros, y luego transforma la cadena en un objeto simbolico.

### Salida

- equation: expresion SymPy equivalente a F(x; theta).
- symbols: diccionario nombre -> simbolo, usado por las etapas posteriores.

### Como interpretar la tabla

La tabla muestra el mapeo entre nombres y simbolos internos. Esta tabla es la base de consistencia semantica del pipeline: si aqui hay un error de nombres, todas las etapas siguientes quedan invalidadas.

In [7]:
equation, symbols = parse_equation(cfg.EQUATION_STRING, cfg.VARIABLES, cfg.PARAMETERS)

print('Ecuacion parseada:', equation, '= 0')
print('Tipo de objeto:', type(equation))
print('Simbolos disponibles:', sorted(list(symbols.keys())))

symbols_df = pd.DataFrame({
    'name': list(symbols.keys()),
    'symbol_repr': [str(v) for v in symbols.values()]
})
display(symbols_df)

Ecuacion parseada: a + x - 2 = 0
Tipo de objeto: <class 'sympy.core.add.Add'>
Simbolos disponibles: ['a', 'x']


,name,symbol_repr
0,x,x
1,a,a


## Etapa 2: Generacion del grid de parametros

### Objetivo

Construir el conjunto de experimentos numericos sobre el dominio de parametros. Cada fila del grid es una tupla theta^(k) sobre la cual luego se resolvera F(x; theta^(k))=0.

### Entrada

- cfg.PARAMETER_RANGES con formato (min, max, n_puntos) por parametro.

### Proceso

La funcion generate_grid produce el producto cartesiano de los valores discretizados de cada parametro. Si hay p parametros con m_i puntos cada uno, el total de tuplas es N = prod(m_i).

### Salida

- grid: matriz N x p con todas las combinaciones.
- param_names: orden canonico de columnas del grid.

### Como interpretar la salida

- Se muestra la tabla completa de tuplas del grid.
- describe().T resume la cobertura estadistica del dominio.

### Criterio de validez

El grid debe cubrir uniformemente el dominio de interes. Si el rango es insuficiente, la regresion simbolica puede aprender expresiones locales y no la ley global.

In [ ]:
grid, param_names = generate_grid(cfg.PARAMETER_RANGES)

print('Cantidad de tuplas de parametros:', grid.shape[0])
print('Cantidad de parametros por tupla:', grid.shape[1])
print('Nombres de parametros:', param_names)

grid_df = pd.DataFrame(grid, columns=param_names)
display(grid_df)
display(grid_df.describe().T)

Grid generado: 50 tuplas (producto cartesiano)
Cantidad de tuplas de parametros: 50
Cantidad de parametros por tupla: 1
Nombres de parametros: ['a']


,a
0,-5.000000
1,-4.795918
2,-4.591837
3,-4.387755
4,-4.183673
5,-3.979592
6,-3.775510
7,-3.571429
8,-3.367347
9,-3.163265


,count,mean,std,min,25%,50%,75%,max
a,50.0,7.105427e-17,2.974975,-5.0,-2.5,0.0,2.5,5.0


## Etapa 3: Resolucion de la ecuacion para cada tupla de parametros

### Objetivo

Obtener, para cada punto del grid de parametros, las raices reales de la ecuacion y construir un dataset numerico confiable para aprendizaje simbolico.

### Entrada

- equation simbolica (Etapa 1).
- var_symbols y param_symbols.
- grid y param_names (Etapa 2).
- parametros de solver desde config: metodo, filtrado complejo y tolerancias.

### Proceso

1. Sustituir cada tupla de parametros en la ecuacion.
2. Resolver F(x; theta)=0 con el solver configurado.
3. Convertir soluciones a numerico.
4. Filtrar soluciones complejas residuales segun tolerancia.
5. Ordenar raices para mantener consistencia de indice por muestra.

### Salida

- results: lista de registros por tupla con parametros, cantidad de raices y valores de raices.
- num_roots_dist: distribucion de cantidad de raices por tupla.
- tabla completa de resultados por tupla.

### Interpretacion de control

- Si len(results) es bajo respecto del grid, el dominio elegido produce pocos casos con raices reales.
- Si la distribucion de num_roots varia mucho, la etapa 4 y 6 deberan manejar ramas parciales o cambios de regimen.
- La tabla completa permite auditar coherencia de parametros y raices en todo el dominio.

In [ ]:
var_symbols = [symbols[v] for v in cfg.VARIABLES]
param_symbols = {p: symbols[p] for p in cfg.PARAMETERS}

results = solve_for_all_parameter_tuples(
    equation,
    var_symbols,
    param_names,
    param_symbols,
    grid,
    method=cfg.SOLVER_METHOD,
    filter_complex=cfg.FILTER_COMPLEX,
    complex_tol=cfg.COMPLEX_TOLERANCE,
    sort_roots=cfg.SORT_ROOTS,
)

print('Tuplas con raices validas:', len(results))

num_roots_dist = {}
for r in results:
    n = r['num_roots']
    num_roots_dist[n] = num_roots_dist.get(n, 0) + 1
print('Distribucion de numero de raices:', num_roots_dist)

results_df = pd.DataFrame([
    {
        **r['parameters'],
        'num_roots': r['num_roots'],
        'roots': r['roots'],
    }
    for r in results
])
results_df.insert(0, 'tuple_index', np.arange(1, len(results_df) + 1))

display(results_df)

Resolviendo ecuación para 50 tuplas de parámetros...


100%|██████████| 50/50 [00:00<00:00, 540.82it/s]

Completado. 50 tuplas con raíces válidas.
Tuplas con raices validas: 50
Distribucion de numero de raices: {1: 50}


,sample,parameters,num_roots,roots
0,1,{'a': -5.0},1,[7.0]
1,2,{'a': -4.795918367346939},1,[6.795918367346939]
2,3,{'a': -4.591836734693878},1,[6.591836734693878]
3,4,{'a': -4.387755102040816},1,[6.387755102040816]
4,5,{'a': -4.183673469387755},1,[6.183673469387755]


## Etapa 4: Agrupacion de raices por rama

### Objetivo

Separar las soluciones en ramas funcionales candidatas g_j(theta), para entrenar una expresion por rama en lugar de mezclar comportamientos distintos.

### Entrada

- results de la etapa 3 (raices ordenadas por tupla).
- param_names para etiquetar X.

### Proceso

La funcion group_by_root_branch construye ramas por posicion ordinal de la raiz en cada tupla: rama 1 (raiz menor), rama 2 (siguiente), etc. Cada rama queda como dataset supervisado (X, y).

### Salida

- branches: lista de ramas.
- Por rama: X (parametros) y y (valores de raiz).
- resumen de tamanos para verificar cobertura por rama.
- tabla completa de puntos para cada rama.

### Riesgo metodologico

Este enfoque asume que la identificacion por orden es consistente en el dominio. Si hay cruces fuertes de ramas, una rama puede mezclar leyes distintas y la regresion simbolica podria requerir varias funciones o expresiones por regiones.

### Interpretacion de tablas

- n_points por rama: cantidad de datos disponibles para aprender.
- X_shape/y_shape: consistencia dimensional del dataset.
- la tabla completa por rama permite inspeccion integral de continuidad y cobertura.

In [ ]:
branches = group_by_root_branch(results, param_names)

print('Numero de ramas detectadas:', len(branches))

branch_summary = []
for i, branch in enumerate(branches, start=1):
    branch_summary.append({
        'branch': i,
        'n_points': int(len(branch['y'])),
        'X_shape': tuple(branch['X'].shape),
        'y_shape': tuple(branch['y'].shape),
    })

display(pd.DataFrame(branch_summary))

for i, branch in enumerate(branches, start=1):
    print(f'\nRama {i} - todos los puntos:')
    branch_df = pd.DataFrame(branch['X'], columns=param_names)
    branch_df['root'] = branch['y']
    display(branch_df)

Raíces agrupadas en 1 ramas:
  Rama 1: 50 puntos
Numero de ramas detectadas: 1


,branch,n_points,X_shape,y_shape
0,1,50,"(50, 1)","(50,)"



Rama 1 - primeros puntos:


,a,root
0,-5.000000,7.000000
1,-4.795918,6.795918
2,-4.591837,6.591837
3,-4.387755,6.387755
4,-4.183673,6.183673


## Etapa 5: Regresion simbolica por rama (PySR + wrapper iterativo)

### Objetivo

Descubrir expresiones analiticas que aproximen o reproduzcan cada rama de raices usando el modulo oficial de regresion simbolica del proyecto.

### Entrada

- branches y param_names de la etapa 4.
- hiperparametros desde config (epsilon, k, niterations, min_points).

### Proceso

La funcion run_for_all_branches ejecuta, para cada rama, el wrapper iterativo de symbolic regression.

De forma simplificada, en cada rama el sistema:
1. Entrena candidatos simbolicos con PySR.
2. Evalua cobertura de puntos matcheados con la tolerancia definida.
3. Acepta funciones utiles y retira puntos explicados.
4. Repite sobre los puntos restantes hasta criterio de parada.

### Salida

- all_results: lista por rama, cada una con 0 o mas funciones descubiertas.
- por funcion: ecuacion candidata y numero de puntos matcheados.

### Interpretacion para tesis

- Si una rama tiene una sola funcion de alta cobertura, su ley es global en el dominio.
- Si aparecen varias funciones por rama, hay cambio de regimen o segmentacion regional.
- Si no aparece ninguna funcion, revisar grid, complejidad permitida, tiempo de busqueda o dificultad intrinseca del caso.

In [ ]:
all_results = run_for_all_branches(
    branches,
    param_names,
    epsilon=cfg.EPSILON,
    k=cfg.K,
    niterations=cfg.NITERATIONS,
    min_points=cfg.MIN_POINTS,
)

summary_rows = []
for branch_idx, branch_results in enumerate(all_results, start=1):
    if len(branch_results) == 0:
        summary_rows.append({
            'branch': branch_idx,
            'function_index': None,
            'equation': None,
            'num_matched': 0,
        })
    else:
        for func_idx, func in enumerate(branch_results, start=1):
            summary_rows.append({
                'branch': branch_idx,
                'function_index': func_idx,
                'equation': str(func['equation']),
                'num_matched': int(func['num_matched']),
            })

sr_summary_df = pd.DataFrame(summary_rows)
display(sr_summary_df)

for branch_idx, branch_results in enumerate(all_results, start=1):
    if len(branch_results) == 0:
        print(f'\nRama {branch_idx}: sin funciones descubiertas.')
        continue

    print(f'\nRama {branch_idx}: detalle completo de funciones y puntos matcheados')
    for func_idx, func in enumerate(branch_results, start=1):
        print(f'  Funcion {func_idx}: {func["equation"]}')
        print(f'  Puntos matcheados: {int(func["num_matched"])}')

        matched_df = pd.DataFrame(func['X_matched'], columns=param_names)
        matched_df['root'] = func['y_matched']
        matched_df.insert(0, 'function_index', func_idx)
        matched_df.insert(0, 'branch', branch_idx)
        display(matched_df)

/home/noel/Disco D/4to_Anno/tesis/Tesis/.venv/lib/python3.12/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...



REGRESIÓN SIMBÓLICA - RAMA 1
Datos: 50 puntos, 1 parámetros
Parámetros: ['a']
Iniciando algoritmo iterativo de regresión simbólica

--- Iteración 1 ---
Puntos restantes: 50

    ── Intento 1/3 ──


[ Info: Started!



Expressions evaluated per second: 3.060e+05
Progress: 667 / 15000 total iterations (4.447%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           8.673e+00  0.000e+00  y = 2
2           4.000e+00  7.740e-01  y = neg(a)
3           1.623e-14  3.314e+01  y = 2 - a
5           6.559e-15  4.531e-01  y = (0.51647 - a) + 1.4835
14          6.275e-15  4.922e-03  y = (((a / a) + ((0.57309 - a) + a)) - a) + 0.42691
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 3.250e+05
Progress: 1444 / 15000 total iterations (9.627%)
════════════════════════════════════════

[ Info: Final population:
[ Info: Results saved to:


    ★ Ecuación #0 matchea 0 pts (complejidad=1, loss=8.673468): 2.000013
    ★ Ecuación #2 matchea 50 pts (complejidad=3, loss=1.2954082e-14): 2.0 - a
    Intento 1: mejor ecuación matchea 50/50 pts
    ★ Nuevo mejor global!
    ✓ ¡100% de cobertura! Saltando intentos restantes.
✓ Ecuación encontrada: 2.0 - a
✓ Puntos matcheados: 50

✓ TODOS LOS PUNTOS FUERON MATCHEADOS!

RESUMEN FINAL
Total de funciones encontradas: 1
Total de puntos matcheados: 50/50
Porcentaje de cobertura: 100.0%


,branch,function_index,equation,num_matched
0,1,1,2.0 - a,50


  - /tmp/tmp70zh0ibj/20260401_205142_A5Wkdr/hall_of_fame.csv


## Etapa 6: Construccion de expresiones finales

### Objetivo

Convertir la salida de regresion simbolica por rama en una representacion final util para reporte: expresion unica, expresion simplificada o modelo por tramos (Piecewise).

### Entrada

- all_results (funciones por rama).
- param_names y param_symbols para construir condiciones simbolicas.

### Proceso

Para cada rama:
1. Si no hay funciones: se marca rama no explicada.
2. Si hay una funcion: se toma como expresion final de rama.
3. Si hay multiples funciones:
   - se intenta simplificar a una sola expresion representativa,
   - se calculan fronteras de region,
   - se intenta construir una expresion Piecewise con condiciones.

### Salida

- final_df con modo de salida por rama: no_function, single_function o multi_function.
- selected_expression: expresion consolidada principal.
- piecewise_expression: expresion por tramos cuando aplica.

### Criterio de cierre del pipeline

El pipeline queda tecnicamente completo cuando cada rama tiene una representacion final interpretable y trazable. Esta tabla es el puente entre el resultado computacional y la redaccion formal de resultados en tesis.

In [12]:
param_symbols = {p: symbols[p] for p in cfg.PARAMETERS}
final_rows = []

for branch_idx, branch_results in enumerate(all_results, start=1):
    if len(branch_results) == 0:
        final_rows.append({
            'branch': branch_idx,
            'mode': 'no_function',
            'n_functions': 0,
            'selected_expression': None,
            'piecewise_expression': None,
        })
        continue

    if len(branch_results) == 1:
        final_rows.append({
            'branch': branch_idx,
            'mode': 'single_function',
            'n_functions': 1,
            'selected_expression': str(branch_results[0]['equation']),
            'piecewise_expression': None,
        })
    else:
        simplified = simplify_to_single_expression(branch_results)

        try:
            piecewise_expr = build_piecewise_expression(branch_results, param_names, param_symbols)
            piecewise_str = str(piecewise_expr)
        except Exception as e:
            piecewise_str = f'Error al construir Piecewise: {type(e).__name__}: {e}'

        boundaries = find_region_boundaries([f['X_matched'] for f in branch_results], param_names)
        print(f'\nRama {branch_idx} - fronteras detectadas:')
        for j, boundary in enumerate(boundaries, start=1):
            print(f'  Funcion {j}: {boundary}')

        final_rows.append({
            'branch': branch_idx,
            'mode': 'multi_function',
            'n_functions': len(branch_results),
            'selected_expression': str(simplified),
            'piecewise_expression': piecewise_str,
        })

final_df = pd.DataFrame(final_rows)
display(final_df)

,branch,mode,n_functions,selected_expression,piecewise_expression
0,1,single_function,1,2.0 - a,None
